In [1]:
import ast
import os
import optuna
import ipynbname
import pandas as pd
import numpy as np
from optuna.exceptions import TrialPruned
#from Functions.AutoCloud_V4_4 import *
from Functions.AutoCloud_V4_5 import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
from Functions.Utils_OPT_R2 import *
import copy

FileName = ipynbname.name()

# Configuração do experimento
FEEDBACK = 'symmetric'    # 'symmetric' (gradiente no passo atual) ou 'random' (ablação: RFLO + DFA)
WTAG, WTAP = False, False # configuração principal: soma ponderada dos grânulos (eq. 36)
EOL = 0.3                 # limiar de fim de vida (EoL) do HI

# rho dinâmico (AutoCloud_V4_2): começa em 0 e é ajustado a cada iteração maximizando V = média(cov)*média(sp)
RHO_TARGET = 'RUL'           # 'HI' (causal, HI um passo à frente) ou 'RUL' (oráculo: usa a RUL real; n_rho previsões por passo)
RHO_MIN, RHO_MAX, N_RHO = 1e-3, 1, 4   # grade log-espaçada de candidatos
LAM = 1.0                   # fator de esquecimento (1.0 = todos os passos com o mesmo peso)

out_path = os.path.join('Optimization', RunTag(FileName,FEEDBACK,WTAG,WTAP), 'multi', 'Optimization.csv')
dfs = [pd.read_csv(out_path)] if os.path.exists(out_path) else []
RS = pd.read_excel(r'Dataset\RS.xlsx')
HI = pd.read_excel(r'Dataset\HI.xlsx')

sig = HI['PC1'].values

c:\Users\Usuario\Desktop\Python\Batteries_Prognostics\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
params = [3.5,28,[70],8,0,1,2,5,19,1,0]
m,nI,nR,nO,mO,N1,N2,N3,τ,mode,act = params
X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)
ref, J, L = RefEoL(sig, EOL, nI, mO, mode)
if len(X) < J: print(f'Aviso: nO={nO} > {nOMax(sig, EOL)}: o fluxo termina antes do EoL (última RUL real = {J-len(X)+1}).')
nI = len(Y[0])
if isinstance(nR, str): nR = ast.literal_eval(nR)

RHO_DYN, RHO_ETA, RHO_T, RHO_BETA = True, 0.1, 0.5, 0.75              # False: rho estático (= RHO), passo, temperatura da cobertura suavizada (escore |e|/e_est), momento
RHO, RHO_MIN, RHO_MAX = 0.5, 1e-2, 10
INTR_STATE = 'reset'       

teda=AutoCloud(m=m,nI=nI,nR=nR,nO=nO+mO,ηS=[N1,N2,N3],mode=mode,act=act,
               tau=τ,eol=EOL,ref=ref,wtaG=WTAG,wtaP=WTAP,feedback=FEEDBACK,mO=mO,
               rho=RHO,lam=LAM,intr_state=INTR_STATE,
               rho_dyn=RHO_DYN,rho_eta=RHO_ETA,rho_T=RHO_T,rho_beta=RHO_BETA,rho_min=RHO_MIN,rho_max=RHO_MAX,
               rls_mode='pol',nf_rls=3,ff_rls=0.9,dt_rls=1e3,
               )

teda.SimGranularRUL(J,L,X,Y,Z)

print(f"HI (HI_Granularity): cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f}")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=3,dsi_mode='hide')


HI (HI_Granularity): cov=0.791 | sp=0.918 | V=média(cov*sp)=0.7635 | Vsum=65.6641


In [24]:
path = r'Dataset\DSIR4\Bearing1_1.csv'
RS1 = pd.read_csv(path).iloc[:,-6:-1]
RS1 = dfMovingAvg(RS1,10)
RS1 = dfExpMovAvg(RS1,0.3)
RS1 = normalize_df(RS1,0,25)

path = r'Dataset\DSIR4\Bearing1_3.csv'
RS2 = pd.read_csv(path).iloc[:,-6:-1]
RS2 = dfMovingAvg(RS2,10)
RS2 = dfExpMovAvg(RS2,0.3)
RS2 = normalize_df(RS2,0,25)

teda=AutoCloud(m=2.0,nI=nI,nR=nR,nO=nO+mO,ηS=[N1,N2,N3],mode=mode,act=act,
               tau=τ,eol=EOL,ref=ref,wtaG=WTAG,wtaP=WTAP,feedback=FEEDBACK,mO=mO,
               rho=RHO,lam=LAM,intr_state=INTR_STATE,
               rho_dyn=RHO_DYN,rho_eta=RHO_ETA,rho_T=RHO_T,rho_beta=RHO_BETA,rho_min=RHO_MIN,rho_max=RHO_MAX,
               rls_mode='pol',nf_rls=3,ff_rls=0.9,dt_rls=1e3,
               )
w = 2
teda.SimGranulation(WindowDSI(RS1,w))   
teda.SimGranulation(WindowDSI(RS2,w))   

print(f"HI (HI_Granularity): cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f}")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide',)

HI (HI_Granularity): cov=0.000 | sp=0.000 | V=média(cov*sp)=0.0000 | Vsum=0.0000


In [32]:
X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)
print(nI, nO, mO, 1, mode)

28 8 0 1 1


In [31]:
PlotFramesPLY([RS])

In [39]:
#print(RS[5:15])
print(HI[25:35])
#print(X[:10])
print(Y[0])
print(Z[0])

         PC1
25  0.854988
26  0.852992
27  0.851020
28  0.849097
29  0.847214
30  0.845366
31  0.843548
32  0.841758
33  0.839995
34  0.838263
[0.91584566 0.91429178 0.91121962 0.90817817 0.90519636 0.90227154
 0.89941598 0.89663373 0.8939275  0.89128272 0.88870303 0.88617045
 0.88366572 0.88121631 0.87881491 0.87645309 0.87413736 0.87187319
 0.86964705 0.86745656 0.86530687 0.86319382 0.86111485 0.85905415
 0.85701525 0.85498778 0.85299248 0.85101983]
[0.84909739 0.84721399 0.84536591 0.84354755 0.84175787 0.83999492
 0.83826309 0.83657452]
